# DVD-JEPA — train a real JEPA world model in <30 seconds

A **Joint-Embedding Predictive Architecture** world model for a bouncing DVD logo.
It predicts the future _in representation space_, learns the physics with no labels,
the future once you add a decoder.

In [3]:
import torch 
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

## 1. The world

A DVD logo bouncing in a 16×16 box. The future is unreadable from one frame
(you can't see velocity) but perfectly predictable (except for noisy pixel fluctuation) from two — so an _observation_
is a stack of two consecutive frames.


In [4]:
from dvd_jepa.world import make_sequences, build_pairs, roll_one
from dvd_jepa.config import FRAME_HEIGHT as H, FRAME_WIDTH as W

frames, pos = make_sequences(num_sequences=4, sequence_length=64, seed=1, noise_std_dev=0.05)
fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for i in range(8):
    ax[i].imshow(frames[0, i * 4], cmap="magma")
    ax[i].axis("off")
    ax[i].set_title(f"t={i * 4}", fontsize=8)
plt.suptitle("one bouncing-logo sequence", y=1.15)
plt.show()

## 2. Train the JEPA

Online encoder + EMA target encoder (stop-grad) + a predictor that matches the
future embedding, plus a VICReg variance term so the representation can't collapse.
Watch `emb-std`: it starts near 0 (collapsing) and is pushed up to ~3 (healthy).


In [5]:
from dvd_jepa.train import (
    train_jepa,
    train_decoder,
    train_baseline,
    linear_probe,
    forecast,
)

frames, _ = make_sequences(seed=0)
obs, nxt = build_pairs(frames)
online, target, predictor = train_jepa(obs, nxt, steps=2500)
dec = train_decoder(target, obs, steps=2000)

## 3. Did it actually learn the world?

Freeze the encoder and fit a **linear** map from the 32-d latent to the true (y, x)
position. Low error ⇒ the latent encodes exact world state, despite never being
told a coordinate.


In [6]:
# build position labels (latest frame of each observation)
_, pos = make_sequences(seed=0)
lbl = torch.tensor(
    np.concatenate([pos[:, t + 1] for t in range(frames.shape[1] - 2)]),
    dtype=torch.float32,
)
_, rmse = linear_probe(target, obs, lbl)
print(f"linear probe recovers position to {rmse:.2f} px in a {H}x{W} box")

## 4. Make it dream (forecast + decode)

Seed from 2 frames, roll the predictor forward **in latent space**, decode each
step to pixels. White = reality, cyan = the dream. It tracks for ~20 steps,
including the wall bounce, then drifts (single-step predictor, errors compound).


In [7]:
vf, _ = roll_one(sequence_length=44, seed=7)
baseline = train_baseline(obs, steps=2000)
jepa_preds, baseline_preds, jepa_errs, baseline_errs = forecast(
    target, predictor, dec, baseline, vf, horizon=24
)
fig, ax = plt.subplots(2, 12, figsize=(13, 2.4))
for k in range(12):
    ax[0, k].imshow(vf[2 * k + 2], cmap="gray")
    ax[0, k].axis("off")
    ax[1, k].imshow(jepa_preds[2 * k], cmap="gray")
    ax[1, k].axis("off")
ax[0, 0].set_ylabel("reality", rotation=0, labelpad=28)
ax[1, 0].set_ylabel("dream", rotation=0, labelpad=24)
plt.suptitle(f"future-frame prediction (mean MSE {np.mean(jepa_errs):.4f})")
plt.show()

## 6. Baseline: MLP

A small MLP that predicts the next frame directly in pixel space —
no latent, no JEPA. Same input (two stacked frames), same MSE loss, same training
budget. Compare: does working in latent space actually help?

In [8]:
device = next(baseline.parameters()).device

# ── single-step MSE comparison ────────────────────────────────────────────────
with torch.no_grad():
    x_all = obs.to(device)
    y_all = obs[:, H * W:].to(device)
    baseline_mse = torch.nn.functional.mse_loss(baseline(x_all), y_all).item()
    jepa_mse = torch.nn.functional.mse_loss(
        dec(target(obs.to(device))), y_all
    ).item()
print(
    f"single-step recon MSE  →  JEPA decoder: {jepa_mse:.5f}  |  MLP baseline: {baseline_mse:.5f}"
)

# ── autoregressive forecast (already computed in section 4) ───────────────────
fig, ax = plt.subplots(3, 12, figsize=(13, 3.6))
for k in range(12):
    ax[0, k].imshow(vf[2 + 2 * k], cmap="gray")
    ax[0, k].axis("off")
    ax[1, k].imshow(jepa_preds[2 * k], cmap="gray")
    ax[1, k].axis("off")
    ax[2, k].imshow(baseline_preds[2 * k], cmap="gray")
    ax[2, k].axis("off")
ax[0, 0].set_ylabel("reality", rotation=0, labelpad=28, fontsize=7)
ax[1, 0].set_ylabel("JEPA", rotation=0, labelpad=32, fontsize=7)
ax[2, 0].set_ylabel("MLP", rotation=0, labelpad=32, fontsize=7)
plt.suptitle(
    f"autoregressive forecast — JEPA MSE {np.mean(jepa_errs):.4f}"
    f"  vs  MLP MSE {np.mean(baseline_errs):.4f}"
    "  (every other step shown)"
)
plt.show()

In [9]:
jepa_params = sum(p.numel() for p in target.parameters()) + sum(
    p.numel() for p in dec.parameters()
)
baseline_params = sum(p.numel() for p in baseline.parameters())
print(
    f"Model size (number of parameters) → JEPA: {jepa_params}  |  MLP baseline: {baseline_params}"
)